In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


file_path = 'WA_Fn-UseC_-Telco-Customer-Churn.csv'

if os.path.exists(file_path):
    print("Loading local file...")
    df = pd.read_csv(file_path)
else:
    print("Local file not found! Downloading from GitHub...")
    import requests
    from io import StringIO
    url = "https://raw.githubusercontent.com/selva86/datasets/master/Customer-Churn.csv"
    response = requests.get(url)
    data = StringIO(response.text)
    df = pd.read_csv(data)
    df.to_csv(file_path, index=False)
    print("File downloaded and saved locally!")

# --- Clean TotalCharges (same as before) ---
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges'])

print(f"Dataset loaded! Shape: {df.shape}")
df.head()

Loading local file...
Dataset loaded! Shape: (7032, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:

df['FamilySize'] = (df['Partner'] == 'Yes').astype(int) + (df['Dependents'] == 'Yes').astype(int)

df['AvgMonthlySpend'] = df['TotalCharges'] / (df['tenure'] + 0.01)  # Add tiny epsilon to avoid division by zero

print("New features created!")
print(df[['FamilySize', 'AvgMonthlySpend', 'Partner', 'Dependents', 'tenure', 'TotalCharges']].head(10))

New features created!
   FamilySize  AvgMonthlySpend Partner Dependents  tenure  TotalCharges
0           1        29.554455     Yes         No       1         29.85
1           0        55.557189      No         No      34       1889.50
2           0        53.805970      No         No       2        108.15
3           0        40.896467      No         No      45       1840.75
4           0        75.447761      No         No       2        151.65
5           0       102.434457      No         No       8        820.50
6           1        88.568832      No        Yes      22       1949.40
7           0        30.159840      No         No      10        301.90
8           1       108.748661     Yes         No      28       3046.05
9           1        56.248186      No        Yes      62       3487.95


In [5]:
# --- SEPARATE X and y ---
y = df['Churn'].map({'Yes': 1, 'No': 0})
X = df.drop(['Churn', 'customerID'], axis=1)  # Drop useless columns

print(f"X shape: {X.shape}")
print(f"y distribution:\n{y.value_counts()}")

X shape: (7032, 21)
y distribution:
Churn
0    5163
1    1869
Name: count, dtype: int64


In [6]:
# --- IDENTIFY COLUMN TYPES ---
# Let's inspect the columns to decide which ones are categorical vs numerical
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)

# --- BUILD PREPROCESSING PIPELINE ---
# Numeric: Scale them
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# Categorical: One-hot encode them (handle_unknown='ignore' handles unseen categories in test set)
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False))
])

# Combine them using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# --- CREATE FULL PIPELINE (Preprocessing + Model) ---
# Using Logistic Regression with class_weight='balanced' to handle imbalance
model = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', model)
])

print("Pipeline created successfully!")
print(pipeline)

Categorical columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Numerical columns: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'AvgMonthlySpend']
Pipeline created successfully!
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['SeniorCitizen', 'tenure',
                                                   'MonthlyCharges',
                                                   'TotalCharges',
                                                   'AvgMonthlySpend']),
                                                 ('cat',
          

In [7]:
# --- SPLIT ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape[0]} rows")
print(f"Test set: {X_test.shape[0]} rows")

# --- FIT THE PIPELINE ---
pipeline.fit(X_train, y_train)

# --- PREDICT ---
y_pred = pipeline.predict(X_test)

# --- EVALUATE ---
print("\n=== PIPELINE PERFORMANCE ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Training set: 5625 rows
Test set: 1407 rows

=== PIPELINE PERFORMANCE ===
Accuracy: 0.7313

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.71      0.80      1033
           1       0.50      0.79      0.61       374

    accuracy                           0.73      1407
   macro avg       0.70      0.75      0.70      1407
weighted avg       0.80      0.73      0.75      1407



In [9]:
# --- COMPARE WITH vs WITHOUT ENGINEERED FEATURES ---

# Remove engineered features from X
X_no_eng = X.drop(['FamilySize', 'AvgMonthlySpend'], axis=1)

# Need to rebuild the pipeline (since column lists change)
numeric_cols_no_eng = X_no_eng.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols_no_eng = X_no_eng.select_dtypes(include=['object', 'category']).columns.tolist()

preprocessor_no_eng = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols_no_eng),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False), categorical_cols_no_eng)
    ])

pipeline_no_eng = Pipeline(steps=[
    ('preprocessor', preprocessor_no_eng),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'))
])

# Split again (same random_state = 42 ensures same split)
X_train_no, X_test_no, y_train_no, y_test_no = train_test_split(X_no_eng, y, test_size=0.2, random_state=42)

# Fit and predict
pipeline_no_eng.fit(X_train_no, y_train_no)
y_pred_no_eng = pipeline_no_eng.predict(X_test_no)

accuracy_no_eng = accuracy_score(y_test_no, y_pred_no_eng)
accuracy_with_eng = accuracy_score(y_test, y_pred)

print("\n=== FEATURE ENGINEERING IMPACT ===")
print(f"Accuracy WITHOUT engineered features: {accuracy_no_eng:.4f}")
print(f"Accuracy WITH engineered features:    {accuracy_with_eng:.4f}")
print(f"Improvement: {accuracy_with_eng - accuracy_no_eng:.4f}")

if accuracy_with_eng > accuracy_no_eng:
    print(" Feature engineering improved performance! Keep them.")
else:
    print(" Feature engineering didn't help much. Try different features next time.")


=== FEATURE ENGINEERING IMPACT ===
Accuracy WITHOUT engineered features: 0.7313
Accuracy WITH engineered features:    0.7313
Improvement: 0.0000
 Feature engineering didn't help much. Try different features next time.


In [10]:
# --- SAVE PIPELINE ---
joblib.dump(pipeline, 'churn_pipeline.joblib')
print("Pipeline saved as 'churn_pipeline.joblib'")

# --- OPTIONAL: Test loading it back ---
loaded_pipeline = joblib.load('churn_pipeline.joblib')
print("Pipeline loaded successfully!")

# Verify it still works
test_pred = loaded_pipeline.predict(X_test)
print(f"Loaded pipeline accuracy: {accuracy_score(y_test, test_pred):.4f}")

Pipeline saved as 'churn_pipeline.joblib'
Pipeline loaded successfully!
Loaded pipeline accuracy: 0.7313


Why Use a Pipeline?

Before this task, I was manually doing:
1. Scale numerical columns
2. One-hot encode categorical columns
3. Split train/test
4. Train model
5. Repeat steps 1-4 when testing new data 

This is dangerous because:
 I might accidentally scale the test set using the test set's own statistics** (data leakage!).
 I might forget to encode a column in the test set that exists in training.
 Code becomes messy and hard to reproduce.

 How the Pipeline Fixes This

My pipeline:
1. ColumnTransformer applies `StandardScaler` to numerical columns and `OneHotEncoder` to categorical columns.
2. The entire pipeline is fit on training data only (`pipeline.fit(X_train, y_train)`).
3. When I call `pipeline.predict(X_test)`, it automatically:
    Uses training set's scaling statistics to transform test data.
    Uses training set's one-hot encoding categories to transform test data.
    Passes the transformed data into the Logistic Regression model.

Bottom Line: This is production-ready ML code that prevents data leakage, is easy to maintain, and can be deployed with a single `joblib.load()` command.